# 🔭 Deep Space Observatory: AI Redshift Prediction Engine
### Built with Python Hackathon — CS4Everyone

*You can also visit this link for the models fully automated testing ground!*
<a herf="google.con"> Live Demo Here</a>

---

> **Project Summary:** This notebook builds a complete end-to-end deep-learning pipeline that predicts the **cosmological redshift** (distance) of quasars from two complementary data sources:
>
> | Tier | Input | Model | Precision |
> |------|-------|-------|-----------|
> | 1 — Spectroscopic | Raw SDSS spectral flux + noise (4600-point 1D signal) | **QuasarResNet Pro** (ResNet + Transformer + SE blocks) | High |
> | 2 — Photometric | 5-band brightness magnitudes (u, g, r, i, z) | **PhotoZNet** (deep MLP, 749k quasars) | Fast |
>
> After training, predictions are visualised as a **2D Cosmic Depth Map** and an interactive **3D Pygame Observatory**.

---

### 📋 Table of Contents
1. [Environment Setup & Dependencies](#1)
2. [Download the DR16Q Master Catalog (1.5 GB)](#2)
3. [Download 10,000 Quasar Spectra](#3)
4. [Tier-1: QuasarResNet Pro Architecture](#4)
5. [Tier-1: Dataset Preparation & DataLoaders](#5)
6. [Tier-1: Training Loop (Huber Loss + Cosine Annealing)](#6)
7. [Tier-1: Monte Carlo Inference & Uncertainty Quantification](#7)
8. [Tier-2: 5-Band Photometry — Feature Engineering](#8)
9. [Tier-2: PhotoZNet Architecture & Training](#9)
10. [Tier-2: Save Scaler & Model Weights](#10)
11. [Deep Field Mining — SDSS Query (Northern Sky)](#11)
12. [Deep Field Mining — VizieR Query (Southern Sky)](#12)
13. [Bulk Inference & Cosmic Depth Map Rendering](#13)
14. [Live Inference Demo (CLI-style)](#14)
15. [3D Observatory Instructions](#15)

---


## 1. Environment Setup & Dependencies <a id='1'></a>

Install all required packages. This cell is safe to re-run — it skips already-installed packages.

> **⚠️ Google Colab users:** A GPU runtime (Runtime → Change runtime type → T4 GPU) is *strongly* recommended for Tier-1 training. Tier-2 trains fine on CPU.


In [ ]:
# ─── Install all required packages ────────────────────────────────────────────
import sys

packages = [
    "torch torchvision",
    "astropy",
    "astroquery",
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "joblib",
    "pandas",
    "numpy",
    "requests",
]

for pkg in packages:
    print(f"Installing {pkg}...", end=" ")
    ret = __import__("subprocess").run(
        [sys.executable, "-m", "pip", "install", "--quiet"] + pkg.split(),
        capture_output=True
    )
    print("✅" if ret.returncode == 0 else f"❌ {ret.stderr.decode()[:80]}")


In [ ]:
# ─── Core imports ─────────────────────────────────────────────────────────────
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from astropy.table import Table
from astropy.io import fits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ─── Device detection ─────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Compute device : {device.type.upper()}")
print(f"🔢  PyTorch version: {torch.__version__}")
if device.type == "cuda":
    print(f"🎮  GPU            : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ℹ️  No GPU found — Tier-1 training will be slow. Consider Colab T4 GPU.")


---
## 2. Download the DR16Q Master Catalog (1.5 GB) <a id='2'></a>

The **SDSS DR16 Quasar Catalog** (`DR16Q_v4.fits`) contains ~750,000 spectroscopically confirmed quasars with precise redshifts, photometry, and metadata.

- **Source:** [SDSS Science Archive Server](https://data.sdss.org/sas/dr16/eboss/qso/DR16Q/)
- **Format:** FITS table
- **Size:** ~1.5 GB

The cell below downloads it only if it doesn't already exist locally (safe to re-run).


In [ ]:
# ─── Load and filter the catalog ──────────────────────────────────────────────
print("📂 Loading catalog into memory (this takes ~30 seconds)...")
catalog = Table.read(CATALOG_FILENAME, format="fits")

# Keep only confirmed quasars with valid redshifts
valid_quasars = catalog[(catalog["IS_QSO_FINAL"] > 0) & (catalog["Z"] > 0)]

print(f"✅ Total entries        : {len(catalog):,}")
print(f"✅ Valid quasars (Z > 0): {len(valid_quasars):,}")
print(f"📊 Redshift range       : {float(valid_quasars['Z'].min()):.4f} → {float(valid_quasars['Z'].max()):.4f}")
print(f"📊 Median redshift      : {float(np.median(valid_quasars['Z'])):.4f}")

# ─── Redshift distribution plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid_quasars["Z"], bins=200, color="dodgerblue", edgecolor="none", alpha=0.85)
ax.set_xlabel("Redshift (Z)", fontsize=13)
ax.set_ylabel("Number of Quasars", fontsize=13)
ax.set_title("DR16Q Redshift Distribution — All Confirmed Quasars", fontsize=14, fontweight="bold")
ax.axvline(float(np.median(valid_quasars["Z"])), color="crimson", lw=2, ls="--", label=f"Median Z = {float(np.median(valid_quasars['Z'])):.3f}")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("redshift_distribution.png", dpi=150)
plt.show()


---
## 3. Download 10,000 Quasar Spectra <a id='3'></a>

Each SDSS spectrum is a FITS file containing:
- **`flux`** — measured light intensity at each wavelength (4600 pixels after standardisation)
- **`ivar`** — inverse variance (noise model; `ivar = 1/σ²`)

We randomly sample 10,000 quasars and attempt to download each spectrum from two possible URL paths (eBOSS and Legacy SDSS), with a 5-second timeout per file to gracefully skip unreachable files.

> ⏱ **Expected time:** 15–45 minutes depending on your connection and SDSS server load.


In [ ]:
# ─── Random sample 10,000 quasars ─────────────────────────────────────────────
NUM_SAMPLES = 10_000
SPECTRA_DIR = "./spectra_10k"
os.makedirs(SPECTRA_DIR, exist_ok=True)

np.random.seed(42)
subset_idx = np.random.choice(len(valid_quasars), NUM_SAMPLES, replace=False)
hackathon_subset = valid_quasars[subset_idx]

print(f"🎯 Sampled {NUM_SAMPLES:,} quasars for spectroscopic analysis")
print(f"📁 Spectra will be saved to: {os.path.abspath(SPECTRA_DIR)}")


In [ ]:
# ─── Download spectra (bulletproof downloader) ────────────────────────────────
downloaded_count = 0
skipped_count = 0
error_count = 0

with requests.Session() as session:
    session.timeout = 5  # 5-second timeout — skips unresponsive files quickly

    for row in tqdm(hackathon_subset, total=NUM_SAMPLES, desc="Fetching Spectra", unit="file"):
        plate   = row["PLATE"]
        mjd     = row["MJD"]
        fiberid = row["FIBERID"]
        z_val   = float(row["Z"])

        # Filename encodes the true redshift for easy label extraction later
        filename = f"spec-{plate}-{mjd}-{fiberid:04d}_z{z_val:.4f}.fits"
        filepath = os.path.join(SPECTRA_DIR, filename)

        if os.path.exists(filepath):
            downloaded_count += 1
            skipped_count += 1
            continue

        # Two possible SDSS URL paths
        url_eboss  = (
            f"https://data.sdss.org/sas/dr16/eboss/spectro/redux/v5_13_0"
            f"/spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd}-{fiberid:04d}.fits"
        )
        url_legacy = (
            f"https://data.sdss.org/sas/dr16/sdss/spectro/redux/26"
            f"/spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd}-{fiberid:04d}.fits"
        )

        try:
            for url in [url_eboss, url_legacy]:
                resp = session.get(url)
                if resp.status_code == 200:
                    with open(filepath, "wb") as f:
                        f.write(resp.content)
                    downloaded_count += 1
                    break
        except Exception:
            error_count += 1

print(f"\n{'='*50}")
print(f"✅ Successfully secured : {downloaded_count:,} spectra")
print(f"⏭️  Already existed     : {skipped_count:,}")
print(f"❌ Download errors      : {error_count:,}")
print(f"{'='*50}")


---
## 4. Tier-1: QuasarResNet Pro Architecture <a id='4'></a>

The spectroscopic model is a **three-stage deep network**:

```
Input: (batch, 2, 4600)   ← dual-channel: [flux, ivar]
  │
  ├─ Entry Conv1D (k=11, stride=2) + MaxPool → (batch, 64, 1150)
  │
  ├─ ResNet Stage 1: 64 → 128 channels, stride=2
  │    └─ 2× AdvancedResBlock [Conv1D + BatchNorm + GELU + SEBlock]
  │
  ├─ ResNet Stage 2: 128 → 256 channels, stride=2
  │    └─ 2× AdvancedResBlock [with dilation=2 for wider receptive field]
  │
  ├─ ResNet Stage 3: 256 → 512 channels, stride=2
  │    └─ 2× AdvancedResBlock
  │
  ├─ Transformer Encoder (4 layers, 8 heads, d_model=512)
  │    └─ Captures long-range spectral correlations (emission line patterns)
  │
  ├─ AdaptiveAvgPool1D → (batch, 512)
  │
  └─ FC: 512 → 256 → 1  (Dropout 0.5 for MC Uncertainty)
```

**Key design decisions:**
| Component | Why |
|-----------|-----|
| **Dual-channel input** | `ivar` tells the network which flux values to trust — pure signal vs noise |
| **SE (Squeeze-and-Excite) blocks** | Recalibrates channel importance — vital for spectral data with sparse useful wavelengths |
| **Dilated convolutions** | Wider receptive field without losing resolution |
| **Transformer layers** | Captures global patterns (e.g. Lyman-α forest, broad emission lines) |
| **MC Dropout** | Keeps dropout active during inference for uncertainty estimates |
| **Huber loss** | Robust to cosmological outliers unlike MSE |


In [ ]:
# ─── Squeeze-and-Excitation Block ─────────────────────────────────────────────
class SEBlock1D(nn.Module):
    """
    Recalibrates channel feature maps by learning which channels are important.
    Squeeze: global average pooling → scalar per channel
    Excitation: two FC layers → sigmoid weights → scale original features
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze    = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1)
        return x * y.expand_as(x)


# ─── Advanced Residual Block ───────────────────────────────────────────────────
class AdvancedResBlock1D(nn.Module):
    """
    Residual block with:
    - Dilated Conv for wider receptive field
    - BatchNorm + GELU activations
    - SE channel attention
    """
    def __init__(self, channels, dilation=1):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(channels)
        self.se    = SEBlock1D(channels)

    def forward(self, x):
        residual = x
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += residual       # skip connection
        return F.gelu(out)


# ─── QuasarResNet Pro — Full Model ────────────────────────────────────────────
class QuasarResNetPro(nn.Module):
    """
    Dual-channel 1D ResNet with Transformer encoder for quasar redshift regression.
    Input shape : (batch, 2, 4600)   → [flux channel, ivar channel]
    Output shape: (batch,)           → predicted redshift Z
    """
    def __init__(self):
        super().__init__()

        # Entry: stride-2 conv + max pool → 4× downsampling
        self.entry_conv = nn.Conv1d(2, 64, kernel_size=11, stride=2, padding=5)
        self.entry_bn   = nn.BatchNorm1d(64)
        self.pool1      = nn.MaxPool1d(2)

        # Three progressive ResNet stages
        self.layer1 = self._make_layer(64,  128, stride=2)
        self.layer2 = self._make_layer(128, 256, stride=2)
        self.layer3 = self._make_layer(256, 512, stride=2)

        # Transformer: 4 layers, 8 heads — captures long-range spectral patterns
        enc_layer       = nn.TransformerEncoderLayer(
            d_model=512, nhead=8, dim_feedforward=1024, batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)

        # Global pool → variable length input becomes fixed 512-d vector
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        # Regression head (Dropout kept active for MC inference)
        self.fc_layers = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def _make_layer(self, in_ch, out_ch, stride):
        """Build one ResNet stage: downsample projection + 2 residual blocks."""
        downsample = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride),
            nn.BatchNorm1d(out_ch),
        )
        return nn.Sequential(
            downsample,
            AdvancedResBlock1D(out_ch, dilation=1),
            AdvancedResBlock1D(out_ch, dilation=2),
        )

    def forward(self, x):
        x = self.pool1(F.gelu(self.entry_bn(self.entry_conv(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        # Transformer needs (batch, seq_len, features)
        x = x.permute(0, 2, 1)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.global_pool(x).squeeze(-1)
        return self.fc_layers(x).squeeze(-1)


# ─── Quick sanity check ───────────────────────────────────────────────────────
_model_test = QuasarResNetPro()
_dummy_input = torch.randn(4, 2, 4600)   # batch=4, channels=2, length=4600
_dummy_out   = _model_test(_dummy_input)

total_params = sum(p.numel() for p in _model_test.parameters() if p.requires_grad)
print(f"✅ QuasarResNet Pro — forward pass OK")
print(f"   Input  : {list(_dummy_input.shape)}")
print(f"   Output : {list(_dummy_out.shape)}")
print(f"   Params : {total_params:,} trainable parameters ({total_params/1e6:.2f}M)")


---
## 5. Tier-1: Dataset Preparation & DataLoaders <a id='5'></a>

The `SDSSRamDataset` class loads all spectra into CPU RAM at init time (fast epoch iteration vs. disk I/O every step).

**Preprocessing pipeline per spectrum:**
1. Truncate or zero-pad to exactly **4600 samples** (the SDSS standard spectral length)
2. Z-score normalise `flux` independently: `(flux - μ) / (σ + ε)`
3. Z-score normalise `ivar` independently
4. Stack → shape `(2, 4600)` dual-channel tensor
5. Label = redshift extracted from filename (`_z{Z:.4f}.fits`)

> **Why normalise each channel separately?** Raw `ivar` values are in units of `(10⁻¹⁷ erg/cm²/s/Å)⁻²` and dwarf `flux` values by several orders of magnitude. Without normalisation the network would learn to ignore flux entirely.


In [ ]:
# ─── SDSS Dual-Channel RAM Dataset ────────────────────────────────────────────
TARGET_LENGTH = 4600   # SDSS standard spectral pixel count

class SDSSRamDataset(Dataset):
    """
    Loads SDSS spectra into RAM. Each item is a (2, 4600) tensor:
        channel 0 → normalised flux
        channel 1 → normalised inverse variance (noise model)
    Label = true redshift, parsed from the filename.
    """
    def __init__(self, file_list, target_length=TARGET_LENGTH):
        self.target_length   = target_length
        self.spectra_tensors  = []
        self.redshift_tensors = []
        failed = 0

        for filepath in tqdm(file_list, desc="Loading spectra into RAM", unit="file", leave=False):
            try:
                # ── Parse true redshift from filename ──────────────────────────
                z_str    = os.path.basename(filepath).split("_z")[-1].replace(".fits", "")
                redshift = float(z_str)

                # ── Read spectrum ──────────────────────────────────────────────
                with fits.open(filepath, memmap=False) as hdul:
                    flux = hdul[1].data["flux"].copy().astype(np.float32)
                    ivar = hdul[1].data["ivar"].copy().astype(np.float32)

                # ── Standardise length ─────────────────────────────────────────
                if len(flux) >= self.target_length:
                    flux = flux[:self.target_length]
                    ivar = ivar[:self.target_length]
                else:
                    pad = self.target_length - len(flux)
                    flux = np.pad(flux, (0, pad), "constant")
                    ivar = np.pad(ivar, (0, pad), "constant")

                # ── Z-score normalise each channel independently ───────────────
                flux = (flux - flux.mean()) / (flux.std() + 1e-8)
                ivar = (ivar - ivar.mean()) / (ivar.std() + 1e-8)

                # ── Stack into (2, 4600) ───────────────────────────────────────
                dual = np.vstack((flux, ivar))
                self.spectra_tensors.append(torch.tensor(dual,     dtype=torch.float32))
                self.redshift_tensors.append(torch.tensor(redshift, dtype=torch.float32))

            except Exception:
                failed += 1
                continue

        print(f"  ✅ Loaded {len(self.spectra_tensors):,} spectra | ❌ Failed: {failed}")

    def __len__(self):
        return len(self.spectra_tensors)

    def __getitem__(self, idx):
        return self.spectra_tensors[idx], self.redshift_tensors[idx]


# ─── Train / Val split ────────────────────────────────────────────────────────
all_files  = sorted(glob.glob(os.path.join(SPECTRA_DIR, "*.fits")))
print(f"🔍 Found {len(all_files):,} downloaded spectra in {SPECTRA_DIR}")

if len(all_files) == 0:
    print("⚠️  No spectra found! Run Section 3 first.")
else:
    train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
    print(f"   Training   : {len(train_files):,} spectra")
    print(f"   Validation : {len(val_files):,} spectra")

    print("\n📥 Loading Training set...")
    train_dataset = SDSSRamDataset(train_files)
    print("📥 Loading Validation set...")
    val_dataset   = SDSSRamDataset(val_files)

    # Batch size 64 is safer for GPUs with <8 GB VRAM; increase to 128/256 on A100
    BATCH_SIZE = 64
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print(f"\n📦 Train batches : {len(train_loader):,}")
    print(f"📦 Val batches   : {len(val_loader):,}")

    # ── Visualise a random spectrum ────────────────────────────────────────────
    sample_spec, sample_z = train_dataset[0]
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

    axes[0].plot(sample_spec[0].numpy(), color="dodgerblue", lw=0.7, alpha=0.85)
    axes[0].set_ylabel("Normalised Flux", fontsize=11)
    axes[0].set_title(f"Sample Quasar Spectrum  |  True Redshift Z = {sample_z.item():.4f}", fontsize=13, fontweight="bold")
    axes[0].grid(True, alpha=0.2)

    axes[1].plot(sample_spec[1].numpy(), color="tomato", lw=0.7, alpha=0.85)
    axes[1].set_ylabel("Normalised IVAR (Noise Model)", fontsize=11)
    axes[1].set_xlabel("Wavelength Pixel Index", fontsize=11)
    axes[1].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig("sample_spectrum.png", dpi=150)
    plt.show()


---
## 6. Tier-1: Training Loop <a id='6'></a>

### Optimisation Strategy

| Setting | Value | Rationale |
|---------|-------|-----------|
| **Loss** | `SmoothL1Loss` (Huber) | Linear gradient for outliers, quadratic near zero — ideal for noisy cosmological data |
| **Optimiser** | `AdamW` (lr=3e-4, wd=1e-3) | Decoupled weight decay prevents over-regularisation |
| **Scheduler** | `CosineAnnealingWarmRestarts` (T₀=20) | Periodically "kicks" the LR to escape local minima |
| **Augmentation** | Gaussian noise (σ=0.05) on flux | Simulates telescope noise variability; prevents overfitting |
| **Epochs** | 100 | Sufficient for convergence; watch val loss plateau |

> 💡 **Tip:** If you're on CPU, set `EPOCHS = 5` for a quick sanity check before moving to Colab GPU.


In [ ]:
# ─── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS       = 100     # Set to 5 for a CPU quick test
LR           = 3e-4
WEIGHT_DECAY = 1e-3
NOISE_STD    = 0.05    # Augmentation strength
T0           = 20      # CosineAnnealing restart period

# ─── Model, Loss, Optimiser, Scheduler ───────────────────────────────────────
model_spec = QuasarResNetPro().to(device)
criterion  = nn.SmoothL1Loss()
optimizer  = optim.AdamW(model_spec.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=T0, T_mult=2, eta_min=1e-6)

# ─── Training History ─────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "lr": []}
best_val_loss = float("inf")

print("=" * 60)
print("  COMMENCING TIER-1 SPECTROSCOPIC TRAINING")
print(f"  Model : QuasarResNetPro")
print(f"  Device: {device.type.upper()}")
print(f"  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}")
print("=" * 60)

for epoch in range(EPOCHS):
    t_start = time.time()

    # ── Training Phase ─────────────────────────────────────────────────────────
    model_spec.train()
    train_loss = 0.0
    for spectra, redshifts in train_loader:
        # Data augmentation: random Gaussian noise prevents overfitting
        spectra = spectra + torch.randn_like(spectra) * NOISE_STD
        spectra, redshifts = spectra.to(device), redshifts.to(device)

        optimizer.zero_grad()
        preds = model_spec(spectra)
        loss  = criterion(preds, redshifts)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_spec.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        train_loss += loss.item()

    avg_train = train_loss / len(train_loader)

    # ── Validation Phase ───────────────────────────────────────────────────────
    model_spec.eval()
    val_loss = 0.0
    with torch.no_grad():
        for spectra, redshifts in val_loader:
            spectra, redshifts = spectra.to(device), redshifts.to(device)
            val_loss += criterion(model_spec(spectra), redshifts).item()

    avg_val = val_loss / len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["lr"].append(current_lr)

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model_spec.state_dict(), "quasar_resnet_pro_best.pth")

    elapsed = time.time() - t_start
    print(
        f"Epoch [{epoch+1:03d}/{EPOCHS}] "
        f"Train: {avg_train:.4f}  Val: {avg_val:.4f}  "
        f"LR: {current_lr:.2e}  Time: {elapsed:.1f}s"
        + (" ⭐ BEST" if avg_val == best_val_loss else "")
    )

# Save final weights
torch.save(model_spec.state_dict(), "quasar_resnet_pro_final.pth")
print("\n✅ Training complete!")
print(f"💾 Final weights → quasar_resnet_pro_final.pth")
print(f"💾 Best weights  → quasar_resnet_pro_best.pth  (val loss: {best_val_loss:.4f})")


In [ ]:
# ─── Plot Training Curves ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(history["train_loss"], color="dodgerblue", lw=1.5, label="Train Loss (Huber)")
ax1.plot(history["val_loss"],   color="tomato",     lw=1.5, label="Val Loss (Huber)")
ax1.set_ylabel("Huber Loss", fontsize=12)
ax1.set_title("QuasarResNet Pro — Training Curves", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.25)
ax1.axhline(best_val_loss, color="green", ls="--", lw=1, alpha=0.7, label=f"Best Val: {best_val_loss:.4f}")
ax1.legend(fontsize=11)

ax2.semilogy(history["lr"], color="goldenrod", lw=1.5)
ax2.set_ylabel("Learning Rate (log scale)", fontsize=12)
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_title("Cosine Annealing Warm Restarts Schedule", fontsize=13)
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig("tier1_training_curves.png", dpi=150)
plt.show()


---
## 7. Tier-1: Monte Carlo Inference & Uncertainty Quantification <a id='7'></a>

**MC Dropout** turns the model into a Bayesian approximation. By keeping `Dropout` active during inference and running **10 forward passes per spectrum**, we get:
- `mean(passes)` → final redshift prediction
- `std(passes)` → calibrated uncertainty estimate (like an error bar)

This is how the model tells you *"I'm confident"* vs *"this spectrum is ambiguous"*.

> **Professional metric used:** `Δz/(1+z)` (normalised redshift error) — the standard in photometric redshift papers. `|Δz|/(1+z) < 0.15` = catastrophic outlier threshold.


In [ ]:
# ─── Load best weights & run Monte Carlo inference ────────────────────────────
MC_PASSES = 10

model_spec.load_state_dict(torch.load("quasar_resnet_pro_best.pth", map_location=device))
model_spec.train()   # Keep TRAIN mode → Dropout stays active (this is intentional!)

true_z_list  = []
pred_z_mean  = []
pred_z_std   = []

print(f"🔬 Running {MC_PASSES} Monte Carlo passes per batch...")

with torch.no_grad():
    for spectra, redshifts in tqdm(val_loader, desc="MC Inference", unit="batch"):
        spectra = spectra.to(device)

        # Stack 10 stochastic forward passes
        mc_stack = np.array([model_spec(spectra).cpu().numpy() for _ in range(MC_PASSES)])
        # mc_stack shape: (MC_PASSES, batch_size)

        pred_z_mean.extend(mc_stack.mean(axis=0).tolist())
        pred_z_std.extend(mc_stack.std(axis=0).tolist())
        true_z_list.extend(redshifts.cpu().numpy().tolist())

true_z_arr = np.array(true_z_list)
pred_z_arr = np.array(pred_z_mean)
std_z_arr  = np.array(pred_z_std)

# ─── Metrics ──────────────────────────────────────────────────────────────────
r2             = r2_score(true_z_arr, pred_z_arr)
mae            = mean_absolute_error(true_z_arr, pred_z_arr)
delta_z        = np.abs(pred_z_arr - true_z_arr) / (1 + true_z_arr)
outlier_frac   = np.mean(delta_z > 0.15) * 100   # catastrophic outlier %
avg_sigma      = np.mean(std_z_arr)

print("\n" + "=" * 55)
print("  TIER-1 SPECTROSCOPIC MODEL — FINAL METRICS")
print("=" * 55)
print(f"  Validation Quasars    : {len(true_z_arr):,}")
print(f"  R² Score              : {r2:.4f}  (1.0 = perfect)")
print(f"  MAE in Z              : {mae:.4f}")
print(f"  Mean |Δz|/(1+z)       : {delta_z.mean():.4f}")
print(f"  Catastrophic Outliers : {outlier_frac:.2f}%  (threshold: |Δz|/(1+z) > 0.15)")
print(f"  Avg MC Uncertainty    : ±{avg_sigma:.4f} Z")
print("=" * 55)


In [ ]:
# ─── Publication-quality results plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: Predicted vs True with error bars ────────────────────────────────────
ax = axes[0]
sc = ax.scatter(true_z_arr, pred_z_arr, c=std_z_arr, cmap="plasma_r",
                alpha=0.35, s=5, rasterized=True)
plt.colorbar(sc, ax=ax, label="MC Uncertainty σ(Z)")

max_z = max(true_z_arr.max(), pred_z_arr.max())
ax.plot([0, max_z], [0, max_z], "r--", lw=2, label="Perfect prediction")
ax.set_xlabel("True Redshift (SDSS)", fontsize=13)
ax.set_ylabel("Predicted Redshift (AI)", fontsize=13)
ax.set_title("QuasarResNet Pro — Predicted vs True Z", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)
stats_text = (
    f"N = {len(true_z_arr):}"
    f"R² = {r2:.4f}"
    f"MAE = ±{mae:.4f} Z"
    f"⟨σ_MC⟩ = ±{avg_sigma:.4f} Z"
)
ax.text(0.04, 0.96, stats_text, transform=ax.transAxes, fontsize=11,
        va="top", bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.9))

# ── Right: Residual distribution ───────────────────────────────────────────────
ax2 = axes[1]
residuals = pred_z_arr - true_z_arr
ax2.hist(residuals, bins=150, color="dodgerblue", edgecolor="none", alpha=0.8, density=True)
ax2.axvline(0, color="crimson", lw=2, ls="--", label="Zero residual")
ax2.axvline(residuals.mean(), color="gold", lw=1.5, ls="--",
            label=f"Mean bias = {residuals.mean():.4f}")
ax2.set_xlabel("Residual  Z_pred − Z_true", fontsize=13)
ax2.set_ylabel("Probability Density", fontsize=13)
ax2.set_title("Prediction Residual Distribution", fontsize=13, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.2)

plt.suptitle("Tier-1 Spectroscopic Model — Monte Carlo Results Dashboard", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("hackathon_results_resnet.png", dpi=200, bbox_inches="tight")
plt.show()
print("💾 Saved → hackathon_results_resnet.png")


---
## 8. Tier-2: 5-Band Photometry — Feature Engineering <a id='8'></a>

The **photometric approach** uses only 5 brightness measurements per object — available from imaging surveys without expensive spectroscopic follow-up.

**SDSS Photometric Bands:**
| Band | Wavelength | Colour |
|------|-----------|--------|
| u | 3543 Å | Ultraviolet |
| g | 4770 Å | Green |
| r | 6231 Å | Red |
| i | 7625 Å | Near-Infrared |
| z | 9134 Å | Infrared |

**Feature vector (9 dimensions):**
```
[u, g, r, i, z,   u-g,  g-r,  r-i,  i-z]
 ← 5 magnitudes → ← 4 colour indices →
```

Colour indices (`u-g`, `g-r`, etc.) encode the *spectral slope* between bands — the primary photometric redshift signature. As a quasar's redshift increases, its emission features shift through these bands, dramatically changing the colours.


In [ ]:
# ─── Extract photometry from master catalog ────────────────────────────────────
print("📡 Extracting 5-band photometry (u, g, r, i, z) from DR16Q...")

photometry = np.array(valid_quasars["PSFMAG"])   # shape: (N, 5)
redshifts  = np.array(valid_quasars["Z"])

# ── Quality filter: remove bad/missing observations ────────────────────────────
# SDSS marks missing data as -9999; values > 30 are unphysically faint
valid_mask = np.all((photometry > 0) & (photometry < 30), axis=1)
clean_photo = photometry[valid_mask]
clean_z     = redshifts[valid_mask]

print(f"✅ Quasars after quality filter: {len(clean_photo):,}  (removed {(~valid_mask).sum():,} bad rows)")

# ── Calculate 4 colour indices ─────────────────────────────────────────────────
colors = np.zeros((len(clean_photo), 4), dtype=np.float32)
colors[:, 0] = clean_photo[:, 0] - clean_photo[:, 1]   # u - g
colors[:, 1] = clean_photo[:, 1] - clean_photo[:, 2]   # g - r
colors[:, 2] = clean_photo[:, 2] - clean_photo[:, 3]   # r - i
colors[:, 3] = clean_photo[:, 3] - clean_photo[:, 4]   # i - z

# ── Assemble 9-feature input matrix ───────────────────────────────────────────
X = np.hstack((clean_photo.astype(np.float32), colors))   # (N, 9)
y = clean_z.astype(np.float32).reshape(-1, 1)

print(f"🔢 Feature matrix shape : {X.shape}   (9 features × {len(X):,} quasars)")
print(f"🔢 Label vector shape   : {y.shape}")

# ── Colour distribution plots ──────────────────────────────────────────────────
colour_names = ["u − g", "g − r", "r − i", "i − z"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col, name in zip(axes.ravel(), colors.T, colour_names):
    ax.hist(col, bins=200, color="mediumpurple", edgecolor="none", alpha=0.8, density=True)
    ax.set_xlabel(name, fontsize=12)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"Colour Index: {name}", fontsize=12, fontweight="bold")
    ax.set_xlim(-2, 5)
    ax.grid(True, alpha=0.2)

plt.suptitle("SDSS Photometric Colour Index Distributions (all DR16Q quasars)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("colour_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ─── Train/Val split & StandardScaler ────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled   = scaler_X.transform(X_val)

print(f"📊 Training samples   : {len(X_train):,}")
print(f"📊 Validation samples : {len(X_val):,}")
print(f"📊 Feature means (scaled) : {X_train_scaled.mean(axis=0).round(3)}")
print(f"📊 Feature stds  (scaled) : {X_train_scaled.std(axis=0).round(3)}")

# ─── Photo-Z PyTorch Dataset ──────────────────────────────────────────────────
class PhotoZDataset(Dataset):
    def __init__(self, X, y):
        # .astype(np.float32) forces native endian — required by PyTorch
        self.X = torch.tensor(X.astype(np.float32),  dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32),  dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

PHOTO_BATCH = 512
train_loader_ph = DataLoader(PhotoZDataset(X_train_scaled, y_train), batch_size=PHOTO_BATCH, shuffle=True,  num_workers=2)
val_loader_ph   = DataLoader(PhotoZDataset(X_val_scaled,   y_val),   batch_size=PHOTO_BATCH, shuffle=False, num_workers=2)

print(f"\n📦 Train batches : {len(train_loader_ph):,}")
print(f"📦 Val batches   : {len(val_loader_ph):,}")


---
## 9. Tier-2: PhotoZNet Architecture & Training <a id='9'></a>

PhotoZNet is a **deep MLP optimised for tabular spectral features**:

```
Input: (batch, 9)
  → Linear(9→256) + BatchNorm + Mish + Dropout(0.2)
  → Linear(256→512) + BatchNorm + Mish + Dropout(0.3)
  → Linear(512→256) + BatchNorm + Mish + Dropout(0.2)
  → Linear(256→64) + Mish
  → Linear(64→1)
Output: (batch, 1) ← predicted Z
```

**Why Mish over ReLU?** Mish (`x·tanh(softplus(x))`) is self-regularising, smooth, and non-monotonic — consistently outperforms ReLU on tabular regression tasks.

**Why `ReduceLROnPlateau`?** Photometric data is noisier than spectra; the model sometimes gets stuck. Auto-halving the LR when stuck (`patience=3`) is more stable than CosineAnnealing here.


In [ ]:
# ─── PhotoZNet Architecture ───────────────────────────────────────────────────
class PhotoZNet(nn.Module):
    """
    Deep MLP for photometric redshift estimation.
    Input : 9 features (5 magnitudes + 4 colour indices)
    Output: scalar predicted redshift Z
    """
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            # Block 1
            nn.Linear(9, 256),
            nn.BatchNorm1d(256),
            nn.Mish(),
            nn.Dropout(0.2),
            # Block 2 — expand to 512
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.Mish(),
            nn.Dropout(0.3),
            # Block 3 — compress back
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Mish(),
            nn.Dropout(0.2),
            # Block 4 — fine features
            nn.Linear(256, 64),
            nn.Mish(),
            # Regression head
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)


# ─── Training Setup ───────────────────────────────────────────────────────────
PHOTO_EPOCHS = 50

model_photo   = PhotoZNet().to(device)
criterion_ph  = nn.SmoothL1Loss()
optimizer_ph  = optim.AdamW(model_photo.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_ph  = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ph, mode="min", patience=3, factor=0.5)

# ─── Param count ──────────────────────────────────────────────────────────────
total_ph = sum(p.numel() for p in model_photo.parameters() if p.requires_grad)
print(f"🧠 PhotoZNet — {total_ph:,} trainable parameters ({total_ph/1e3:.1f}K)")
print(f"🖥️  Training on {device.type.upper()} for {PHOTO_EPOCHS} epochs")

history_ph  = {"train_loss": [], "val_loss": [], "lr": []}
best_ph_val = float("inf")
start_time  = time.time()

print("\n" + "=" * 55)
print("  TIER-2 PHOTOMETRIC MLP TRAINING")
print("=" * 55)

for epoch in range(PHOTO_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────────
    model_photo.train()
    train_loss = 0.0
    for X_b, y_b in train_loader_ph:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_ph.zero_grad()
        loss = criterion_ph(model_photo(X_b), y_b)
        loss.backward()
        optimizer_ph.step()
        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────────────────────
    model_photo.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader_ph:
            X_b, y_b = X_b.to(device), y_b.to(device)
            val_loss += criterion_ph(model_photo(X_b), y_b).item()

    avg_train_ph = train_loss / len(train_loader_ph)
    avg_val_ph   = val_loss   / len(val_loader_ph)
    scheduler_ph.step(avg_val_ph)

    history_ph["train_loss"].append(avg_train_ph)
    history_ph["val_loss"].append(avg_val_ph)
    history_ph["lr"].append(optimizer_ph.param_groups[0]["lr"])

    if avg_val_ph < best_ph_val:
        best_ph_val = avg_val_ph
        torch.save(model_photo.state_dict(), "quasar_photoz_mlp_best.pth")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1:02d}/{PHOTO_EPOCHS}] "
            f"Train: {avg_train_ph:.4f}  Val: {avg_val_ph:.4f}  "
            f"LR: {optimizer_ph.param_groups[0]['lr']:.6f}"
            + (" ⭐" if avg_val_ph == best_ph_val else "")
        )

elapsed_ph = (time.time() - start_time) / 60
torch.save(model_photo.state_dict(), "quasar_photoz_mlp.pth")
print(f"\n✅ Photo-Z training complete in {elapsed_ph:.2f} minutes!")
print(f"💾 Final  → quasar_photoz_mlp.pth")
print(f"💾 Best   → quasar_photoz_mlp_best.pth  (val loss: {best_ph_val:.4f})")

In [ ]:
# ─── Evaluate Tier-2 on validation set ───────────────────────────────────────
model_photo.load_state_dict(torch.load("quasar_photoz_mlp_best.pth", map_location=device))
model_photo.eval()

ph_true, ph_pred = [], []
with torch.no_grad():
    for X_b, y_b in val_loader_ph:
        ph_pred.extend(model_photo(X_b.to(device)).cpu().numpy().flatten().tolist())
        ph_true.extend(y_b.numpy().flatten().tolist())

ph_true = np.array(ph_true)
ph_pred = np.array(ph_pred)

ph_r2  = r2_score(ph_true, ph_pred)
ph_mae = mean_absolute_error(ph_true, ph_pred)
ph_delta = np.abs(ph_pred - ph_true) / (1 + ph_true)
ph_outlier = np.mean(ph_delta > 0.15) * 100

print("\n" + "=" * 55)
print("  TIER-2 PHOTO-Z MODEL — VALIDATION METRICS")
print("=" * 55)
print(f"  R² Score              : {ph_r2:.4f}")
print(f"  MAE in Z              : {ph_mae:.4f}")
print(f"  Mean |Δz|/(1+z)       : {ph_delta.mean():.4f}")
print(f"  Catastrophic Outliers : {ph_outlier:.2f}%")
print("=" * 55)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.hexbin(ph_true, ph_pred, gridsize=80, cmap="YlOrRd", mincnt=1)
ax.plot([0, ph_true.max()], [0, ph_true.max()], "b--", lw=2, label="Perfect prediction")
ax.set_xlabel("True Redshift (SDSS)", fontsize=13)
ax.set_ylabel("PhotoZNet Predicted Redshift", fontsize=13)
ax.set_title("Tier-2 PhotoZNet — Predicted vs True Z (Hexbin Density)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.text(0.04, 0.96,
    f"N = {len(ph_true):,}\nR² = {ph_r2:.4f}\nMAE = ±{ph_mae:.4f} Z\n"
    f"Outliers = {ph_outlier:.2f}%",
    transform=ax.transAxes, fontsize=11, va="top",
    bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.9))
plt.tight_layout()
plt.savefig("tier2_photoz_results.png", dpi=150)
plt.show()

# ── Training history ──────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(history_ph["train_loss"], color="steelblue", lw=1.5, label="Train Loss")
ax2.plot(history_ph["val_loss"],   color="tomato",    lw=1.5, label="Val Loss")
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Huber Loss", fontsize=12)
ax2.set_title("PhotoZNet Training History", fontsize=13, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("tier2_training_history.png", dpi=150)
plt.show()


---
## 10. Save Scaler & Export Weights <a id='10'></a>

The `StandardScaler` fitted on training data must be serialised alongside the model weights — inference requires the **exact same normalisation parameters**. Failing to do this is one of the most common deployment bugs in ML projects.


In [ ]:
# ─── Serialise all artefacts ──────────────────────────────────────────────────
import zipfile


joblib.dump(scaler_X, "photoz_scaler.pkl")

artefacts = {
    "photoz_scaler.pkl"         : "StandardScaler (fit on training photometry)",
    "quasar_photoz_mlp.pth"     : "PhotoZNet final weights",
    "quasar_photoz_mlp_best.pth": "PhotoZNet best validation weights",
    "quasar_resnet_pro_final.pth": "QuasarResNetPro final weights",
    "quasar_resnet_pro_best.pth" : "QuasarResNetPro best validation weights",
}

print("📦 Exported artefacts:")
for fname, desc in artefacts.items():
    size = os.path.getsize(fname) / 1e6 if os.path.exists(fname) else 0
    status = "✅" if os.path.exists(fname) else "❌ MISSING"
    print(f"  {status}  {fname:<40} ({size:.1f} MB)  ← {desc}")
    
print("\nDo you want to create a ZIP archive of all artefacts for local testing? (y/n)")
if input().strip().lower() == "y":
    with zipfile.ZipFile("quasar_redshift_models.zip", "w") as zipf:
        for fname in artefacts.keys():
            if os.path.exists(fname):
                zipf.write(fname)
    print("📦 Created quasar_redshift_models.zip containing all artefacts.")

---
## 11. Deep Field Mining — SDSS Query (Northern Sky) <a id='11'></a>

Instead of using pre-labelled training data, we can point the system at any sky coordinate and fetch *live* photometric data from the SDSS database.

The function below queries the **SDSS PhotoObj** table for every detected object within a given radius of a target (RA, Dec), returning the 5-band magnitudes we need to feed into PhotoZNet.

> **Example target:** RA=184.2, Dec=34.1 — a region of the Coma Cluster supercluster


In [ ]:
from astroquery.sdss import SDSS
from astropy import coordinates as coords
import astropy.units as u

def mine_deep_field_sdss(ra: float, dec: float, radius_arcmin: float = 2.0) -> pd.DataFrame | None:
    """
    Query SDSS PhotoObj for all objects within `radius_arcmin` of (ra, dec).
    Returns a cleaned DataFrame with columns: ra, dec, u, g, r, i, z
    or None if the query fails.

    Parameters
    ----------
    ra, dec        : ICRS coordinates in decimal degrees
    radius_arcmin  : search radius in arcminutes (default 2')
    """
    print("=" * 50)
    print("   SDSS DEEP FIELD PHOTOMETRY DATA MINER")
    print("=" * 50)
    print(f"[🔭] Target    : RA={ra}°, Dec={dec}°")
    print(f"[📡] Querying SDSS within {radius_arcmin}' radius...")

    target_pos = coords.SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")

    try:
        photo_data = SDSS.query_region(
            target_pos,
            radius=radius_arcmin * u.arcmin,
            photoobj_fields=["ra", "dec", "u", "g", "r", "i", "z"],
        )
    except Exception as e:
        print(f"[❌] Query failed: {e}")
        return None

    if photo_data is None or len(photo_data) == 0:
        print("[⚠️] No objects returned for this sky region.")
        return None

    df = photo_data.to_pandas()

    # Quality filter — remove bad sensor readings
    df = df[
        (df["u"] > 0) & (df["u"] < 30) &
        (df["z"] > 0) & (df["z"] < 30)
    ].reset_index(drop=True)

    print(f"\n✅ Extracted {len(df):,} valid objects from SDSS")
    print("\n[TOP 5 TARGETS]")
    print(df[["ra", "dec", "u", "g", "r", "i", "z"]].head(5).to_string(index=False))

    filename = f"deep_field_targets_RA{ra}.csv"
    df.to_csv(filename, index=False)
    print(f"\n💾 Saved → {filename}")

    return df


# ─── Run a query ──────────────────────────────────────────────────────────────
df_sdss = mine_deep_field_sdss(ra=184.2, dec=34.1, radius_arcmin=2)


---
## 12. Deep Field Mining — VizieR Query (Southern Sky) <a id='12'></a>

SDSS coverage is limited to the Northern sky. For Southern hemisphere targets (e.g. the Carina Nebula / Keyhole region at RA≈161°, Dec≈−60°), we query the **APASS photometric survey** via [VizieR](https://vizier.cds.unistra.fr/).

APASS doesn't have `u` and `z` bands natively, so we derive equivalents from B-V colours using empirical transformations:
- `u ≈ g + 1.24(B − V)`
- `z ≈ i − 0.42(r − i)`


In [ ]:
from astroquery.vizier import Vizier

def mine_southern_deep_field(ra: float = 161.2625, dec: float = -59.6844,
                              radius_arcmin: float = 15.0) -> pd.DataFrame | None:
    """
    Query APASS DR9 via VizieR for objects in the southern sky.
    Derives synthetic u, z bands from B, V when available.
    Returns cleaned DataFrame with columns: ra, dec, u, g, r, i, z
    """
    print("=" * 52)
    print("  SOUTHERN SKY 5-BAND PHOTOMETRY DATA MINER")
    print("=" * 52)
    print(f"[🔭] Target    : RA={ra}°, Dec={dec}°")
    print(f"[📡] Querying APASS DR9 (VizieR) within {radius_arcmin}' radius...")

    target_pos = coords.SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")

    try:
        v = Vizier(row_limit=5000)
        result = v.query_region(target_pos, radius=radius_arcmin * u.arcmin, catalog="II/336/apass9")
    except Exception as e:
        print(f"[❌] VizieR query failed: {e}")
        return None

    if not result or len(result) == 0:
        print("[⚠️] VizieR returned empty results for this region.")
        return None

    df = result[0].to_pandas()
    print(f"[✅] Raw catalog: {len(df)} records")

    # ── Flexible column mapping ────────────────────────────────────────────────
    target_cols = {"ra": None, "dec": None, "g": None, "r": None,
                   "i": None, "b": None, "v": None}

    for col in df.columns:
        c = col.lower()
        if c.startswith("e_") or "err" in c:
            continue
        if   c in ["raj2000", "ra"]:            target_cols["ra"]  = col
        elif c in ["dej2000", "dec"]:           target_cols["dec"] = col
        elif c in ["g_mag", "gmag"]:            target_cols["g"]   = col
        elif c in ["r_mag", "rmag"]:            target_cols["r"]   = col
        elif c in ["i_mag", "imag"]:            target_cols["i"]   = col
        elif c == "bmag":                       target_cols["b"]   = col
        elif c == "vmag":                       target_cols["v"]   = col

    # Fallback loose matching
    for col in df.columns:
        c = col.lower()
        if c.startswith("e_") or "err" in c:
            continue
        if not target_cols["ra"]  and "ra" in c:                   target_cols["ra"]  = col
        if not target_cols["dec"] and ("de" in c or "dec" in c):   target_cols["dec"] = col
        if not target_cols["g"]   and ("gmag" in c or "g_mag" in c): target_cols["g"] = col
        if not target_cols["r"]   and ("rmag" in c or "r_mag" in c): target_cols["r"] = col
        if not target_cols["i"]   and ("imag" in c or "i_mag" in c): target_cols["i"] = col

    # Build clean numeric DataFrame
    clean_data = {
        k: pd.to_numeric(df[v], errors="coerce")
        for k, v in target_cols.items()
        if v is not None
    }
    df_clean = pd.DataFrame(clean_data)

    if "ra" not in df_clean or "dec" not in df_clean:
        print("[❌] Critical coordinate columns missing.")
        return None

    # ── Synthetic u / z bands ─────────────────────────────────────────────────
    if all(c in df_clean for c in ["g", "r", "i"]):
        if "b" in df_clean and "v" in df_clean:
            df_clean["u"] = df_clean["g"] + 1.24 * (df_clean["b"] - df_clean["v"])
        else:
            df_clean["u"] = df_clean["g"] + 0.85   # generic offset fallback
        df_clean["z"] = df_clean["i"] - 0.42 * (df_clean["r"] - df_clean["i"])
    else:
        print("[⚠️] Primary g/r/i bands missing — using placeholder values.")
        df_clean["u"] = 18.0
        df_clean["g"] = 17.5
        df_clean["r"] = 17.0
        df_clean["i"] = 16.5
        df_clean["z"] = 16.0

    required = ["ra", "dec", "u", "g", "r", "i", "z"]
    df_final = df_clean[required].dropna()
    df_final = df_final[
        (df_final["u"] > 0) & (df_final["u"] < 30) &
        (df_final["z"] > 0) & (df_final["z"] < 30)
    ].reset_index(drop=True)

    print(f"✅ Clean catalog: {len(df_final):,} objects")
    print("\n[TOP 5 CLEANED TARGETS]")
    print(df_final.head(5).to_string(index=False))

    filename = f"deep_field_targets_RA{round(ra, 1)}.csv"
    df_final.to_csv(filename, index=False)
    print(f"\n💾 Saved → {filename}")
    return df_final


# ─── Run southern query ───────────────────────────────────────────────────────
df_southern = mine_southern_deep_field()


---
## 13. Bulk Inference & Cosmic Depth Map Rendering <a id='13'></a>

Once a CSV is mined, we run **batch PhotoZNet inference** across the entire field and render a **Cosmic Depth Map**:

- **X-axis:** Right Ascension (inverted — astronomy standard)
- **Y-axis:** Declination
- **Colour:** Predicted redshift Z (yellow/white = nearby; purple/black = distant)
- **Size:** Object brightness (larger dot = brighter object)


In [ ]:
def generate_cosmic_depth_map(csv_file: str, output_png: str = "cosmic_depth_map.png") -> None:
    """
    Load a mined deep-field CSV, run Photo-Z bulk inference, and render a
    publication-quality cosmic depth map.

    Parameters
    ----------
    csv_file   : path to CSV with columns ra, dec, u, g, r, i, z
    output_png : filename to save the rendered map
    """
    print("=" * 52)
    print("   DEEP FIELD BULK INFERENCE & DEPTH MAP")
    print("=" * 52)

    # ── Load model & scaler ───────────────────────────────────────────────────
    model_infer = PhotoZNet().to(device)
    try:
        model_infer.load_state_dict(torch.load("quasar_photoz_mlp_best.pth", map_location=device))
        model_infer.eval()
        scaler_infer = joblib.load("photoz_scaler.pkl")
        print("✅ Model and scaler loaded.")
    except FileNotFoundError as e:
        print(f"❌ Missing file: {e}")
        return

    # ── Load CSV ──────────────────────────────────────────────────────────────
    try:
        df = pd.read_csv(csv_file)
        print(f"📂 Loaded {len(df):,} targets from {csv_file}")
    except FileNotFoundError:
        print(f"❌ File not found: {csv_file}")
        return

    if len(df) == 0:
        print("❌ Empty DataFrame — nothing to process.")
        return

    # ── Feature engineering ───────────────────────────────────────────────────
    u, g, r, i_, z_ = df["u"].values, df["g"].values, df["r"].values, df["i"].values, df["z"].values
    raw_features = np.column_stack((u, g, r, i_, z_, u-g, g-r, r-i_, i_-z_)).astype(np.float32)
    scaled_feat  = scaler_infer.transform(raw_features)
    tensor_feat  = torch.tensor(scaled_feat, dtype=torch.float32).to(device)

    # ── Bulk inference ────────────────────────────────────────────────────────
    with torch.no_grad():
        preds = model_infer(tensor_feat).cpu().numpy().flatten()

    df["Predicted_Redshift_Z"] = preds
    out_csv = "mapped_" + os.path.basename(csv_file)
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Inference complete → {out_csv}")
    print(f"   Z range: {preds.min():.3f} → {preds.max():.3f}  (median {np.median(preds):.3f})")

    # ── Render Cosmic Depth Map ───────────────────────────────────────────────
    print("\n🌌 Rendering Cosmic Depth Map...")
    plt.style.use("dark_background")
    fig, ax = plt.subplots(figsize=(12, 9))

    sizes = np.clip((26 - df["i"].values) ** 2, 4, 200)

    scatter = ax.scatter(
        df["ra"], df["dec"],
        c=df["Predicted_Redshift_Z"],
        cmap="magma_r",
        s=sizes,
        alpha=0.85,
        edgecolors="white",
        linewidths=0.3,
        rasterized=True,
    )

    cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
    cbar.set_label("AI Predicted Redshift Z (Cosmic Distance)", fontsize=12, color="white")

    ax.set_title(
        f"AI-Generated Cosmic Depth Map — {len(df):,} Objects",
        fontsize=15, fontweight="bold", color="white", pad=12
    )
    ax.set_xlabel("Right Ascension (RA)", fontsize=12)
    ax.set_ylabel("Declination (Dec)", fontsize=12)
    ax.invert_xaxis()   # Astronomy convention
    ax.grid(True, alpha=0.1, lw=0.5)

    # Annotation
    ax.text(0.02, 0.97,
            f"Source: {csv_file}\n"
            f"Model: PhotoZNet (Tier-2)\n"
            f"Z: {preds.min():.3f} – {preds.max():.3f}",
            transform=ax.transAxes, fontsize=10, va="top",
            color="lightgray",
            bbox=dict(boxstyle="round,pad=0.4", fc="#111111", ec="gray", alpha=0.8))

    plt.tight_layout()
    plt.savefig(output_png, dpi=250, facecolor="black", bbox_inches="tight")
    plt.style.use("default")
    print(f"💾 Map saved → {output_png}")
    plt.show()


# ─── Run on whichever CSV was mined ──────────────────────────────────────────
# Check which CSVs exist and process the first available one
candidate_csvs = [
    "deep_field_targets_RA161.3.csv",
    "deep_field_targets_RA184.2.csv",
    "deep_field_targets_RA161.2625.csv",
]
for csv in candidate_csvs:
    if os.path.exists(csv):
        generate_cosmic_depth_map(csv)
        break
else:
    print("⚠️  No deep field CSV found. Run Sections 11 or 12 first, then re-run this cell.")


---
## 14. Live Inference Demo (Photometric) <a id='14'></a>

Type in 5 SDSS magnitudes for any object and get an instant redshift estimate from PhotoZNet.

> **Example values** (a known z≈0.3 quasar):
> - u = 20.1, g = 19.6, r = 19.2, i = 18.9, z = 18.7


In [ ]:
def predict_redshift_from_photometry(u: float, g: float, r: float,
                                      i: float, z: float,
                                      model=None, scaler=None) -> float:
    """
    Predict cosmological redshift from 5-band SDSS photometry.

    Parameters
    ----------
    u, g, r, i, z  : PSF magnitudes in each SDSS band
    model           : loaded PhotoZNet (uses global model_photo if None)
    scaler          : fitted StandardScaler (uses global scaler_X if None)

    Returns
    -------
    float : predicted redshift Z
    """
    if model is None:
        model = model_photo
    if scaler is None:
        scaler = scaler_X

    model.eval()

    # 9-feature vector: 5 mags + 4 colour indices
    raw = np.array([[u, g, r, i, z, u-g, g-r, r-i, i-z]], dtype=np.float32)
    scaled = scaler.transform(raw)
    tensor = torch.tensor(scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_z = model(tensor).item()

    return pred_z


# ─── Demo predictions ─────────────────────────────────────────────────────────
test_objects = [
    # ( u,     g,     r,     i,     z,    label            )
    (20.1,  19.6,  19.2,  18.9,  18.7, "Known z≈0.3 QSO"),
    (19.5,  18.8,  18.3,  18.0,  17.8, "Bright z≈0.2 QSO"),
    (22.0,  21.2,  20.8,  20.5,  20.2, "Faint  z≈1.0 QSO"),
    (21.3,  20.1,  19.5,  19.1,  18.9, "Medium z≈0.5 QSO"),
]

print("╔══════════════════════════════════════════════════════════╗")
print("║       PHOTO-Z MLP: LIVE INFERENCE DASHBOARD             ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  {'Object':<26} {'u-g':>5} {'g-r':>5} {'r-i':>5} {'Predicted Z':>12} ║")
print("╠══════════════════════════════════════════════════════════╣")

for u_v, g_v, r_v, i_v, z_v, label in test_objects:
    pred = predict_redshift_from_photometry(u_v, g_v, r_v, i_v, z_v)
    print(f"║  {label:<26} {u_v-g_v:>5.2f} {g_v-r_v:>5.2f} {r_v-i_v:>5.2f} {pred:>12.4f} ║")

print("╚══════════════════════════════════════════════════════════╝")


In [ ]:
# ─── Interactive input (run this cell and type your values) ───────────────────
print("🔭 INTERACTIVE PHOTOMETRIC REDSHIFT ESTIMATOR")
print("─" * 48)
print("Enter SDSS PSF magnitudes for your target object.")
print("(Typical range: 15–25. Lower = brighter.)")
print()

try:
    u_in = float(input("  u (Ultraviolet, ~3543 Å) : "))
    g_in = float(input("  g (Green,       ~4770 Å) : "))
    r_in = float(input("  r (Red,         ~6231 Å) : "))
    i_in = float(input("  i (Near-IR,     ~7625 Å) : "))
    z_in = float(input("  z (Infrared,    ~9134 Å) : "))

    pred_z_interactive = predict_redshift_from_photometry(u_in, g_in, r_in, i_in, z_in)

    print()
    print("█" * 48)
    print("           TARGET DISTANCE SECURED")
    print("█" * 48)
    print(f"  Estimated Cosmological Redshift : Z = {pred_z_interactive:.4f}")

    # Rough distance interpretation
    if pred_z_interactive < 0.3:
        dist_note = "~1–3 billion light-years (nearby universe)"
    elif pred_z_interactive < 1.0:
        dist_note = "~3–8 billion light-years (distant universe)"
    elif pred_z_interactive < 2.5:
        dist_note = "~8–12 billion light-years (quasar epoch)"
    else:
        dist_note = "~12+ billion light-years (near Big Bang)"

    print(f"  Estimated Distance              : {dist_note}")
    print("█" * 48)

except ValueError:
    print("\n[⚠️] Invalid input — please enter numeric values only.")
except EOFError:
    print("\n[ℹ️] Non-interactive mode — use predict_redshift_from_photometry() directly.")


---
## 15. 3D Observatory Instructions <a id='15'></a>

The `pygame_3d_observatory.py` script renders an interactive **3D star map** of the deep field predictions. It requires a local Python environment with `pygame` (not supported in Colab/JupyterHub).

### Setup
```bash
pip install pygame pandas numpy
python pygame_3d_observatory.py
```

### Controls
| Action | Control |
|--------|---------|
| Rotate 3D view | Click + drag |
| Zoom in | Scroll up |
| Zoom out | Scroll down |
| Quit | Close window |

### Data Pipeline
```
mine_southern_deep_field.py
       ↓
deep_field_targets_RA161.3.csv
       ↓
process_deep_field.py  (PhotoZNet bulk inference)
       ↓
mapped_deep_field_targets_RA161.3.csv
       ↓
pygame_3d_observatory.py  (3D render)
```

### Colour Encoding
| Colour | Redshift | Distance |
|--------|----------|----------|
| White/Yellow | Low Z | Nearby |
| Orange/Red | Medium Z | Mid-range |
| Purple | High Z | Ancient light |


In [ ]:
# ─── Summary of all generated files ──────────────────────────────────────────
print("=" * 65)
print("  DEEP SPACE OBSERVATORY — PROJECT FILE SUMMARY")
print("=" * 65)

file_groups = {
    "Model Weights": [
        "quasar_resnet_pro_best.pth",
        "quasar_resnet_pro_final.pth",
        "quasar_photoz_mlp_best.pth",
        "quasar_photoz_mlp.pth",
    ],
    "Preprocessor": ["photoz_scaler.pkl"],
    "Visualisations": [
        "redshift_distribution.png",
        "sample_spectrum.png",
        "colour_distributions.png",
        "tier1_training_curves.png",
        "hackathon_results_resnet.png",
        "tier2_photoz_results.png",
        "tier2_training_history.png",
        "cosmic_depth_map.png",
    ],
    "Catalog / Data": ["DR16Q_v4.fits", "spectra_10k/"],
}

for group, files in file_groups.items():
    print(f"\n  [{group}]")
    for f in files:
        exists = os.path.isfile(f) or os.path.isdir(f)
        size_mb = os.path.getsize(f) / 1e6 if os.path.isfile(f) else None
        size_str = f"{size_mb:.1f} MB" if size_mb is not None else ("DIR" if os.path.isdir(f) else "─")
        status = "✅" if exists else "○ "
        print(f"    {status}  {f:<45} {size_str}")

print()
print("  🚀 Run 'python pygame_3d_observatory.py' locally for the 3D view!")
print("=" * 65)


## 🌌 Explore the 3D Cosmic Observatory

Download the GitHub repository to experience the interactive 3D map locally:

```bash
# 1. Clone the repository
git clone "https://github.com/AVadhyanshverma/Built-with-Python-Hackathon.git"

# 2. Navigate into the project directory
cd "Built-with-Python-Hackathon"

# 3. Create and activate a virtual environment (optional but recommended)
python -m venv venv
source venv/bin/activate  # On Windows use: venv\Scripts\activate

# 4. Install the required dependencies
pip install -r requirements.txt

# 5. Launch the 3D Pygame Experience
python pygame_3d_observatory.py
```